In [ ]:
import os

os.chdir(r"C:\Users\yoshi\Downloads\PandasAssignment")
print(os.getcwd())

In [ ]:
#DATA INGESTION

#Importing the dataset and printing the first 5 rows using head func
import pandas as pd
df = pd.read_csv("household_power_consumption.txt", sep=";")

print(df.head())

In [ ]:
#Adding timestamp 
#Initially only used DataFrame methods then used datetime func to turn into objects so python can undertsand it 
#Also learnt formatting here

df["timestamp"] = pd.to_datetime(df["Date"] + " " + df["Time"],format="%d/%m/%Y %H:%M:%S")
print(df.head())

In [ ]:
#before removing NULL vals, checking missing mixed values as per wrning initially
print((df == "?").sum())

In [ ]:
#Checking data set size to verify after cleaning up
print(df.shape)

In [ ]:
df = df.replace("?", pd.NA) #turn missing to Null
df = df.dropna()

print(df.shape)  #check if we got rid of the null vals

In [ ]:
print(df.info())

In [ ]:
#Had to fix data types to run aggregate funcs more easily
df["Global_active_power"] = pd.to_numeric(df["Global_active_power"])
df["Global_reactive_power"] = pd.to_numeric(df["Global_reactive_power"])
df["Voltage"] = pd.to_numeric(df["Voltage"])
df["Global_intensity"] = pd.to_numeric(df["Global_intensity"])
df["Sub_metering_1"] = pd.to_numeric(df["Sub_metering_1"])
df["Sub_metering_2"] = pd.to_numeric(df["Sub_metering_2"])

In [ ]:
#verifying the data types
print(df.info())

In [ ]:
import sys
print(sys.executable)

In [ ]:
import sys
!{sys.executable} -m pip install pymongo

In [ ]:
import pymongo
print(pymongo.__version__)

In [ ]:
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017")
print(client.list_database_names())

In [ ]:
db = client["power_db"]
collection = db["power_consumption"]
print(collection)

In [ ]:
#new dataframe with 5 rows only rn
sample_df = df.head(5)

In [ ]:
records = sample_df.to_dict("records")

In [ ]:
#Convert to dictionary to be more compatibke to mongodb
records = sample_df.to_dict("records")

In [ ]:
#Inserting into mongodb
result = collection.insert_many(records)

In [ ]:
#Verifying if the insertion worked or nah
print(len(result.inserted_ids))

In [ ]:
peak_row = df.loc[df["Global_active_power"].idxmax()]

print(peak_row["timestamp"])
print(peak_row["Global_active_power"])

In [ ]:
print(collection.count_documents({}))

In [ ]:
#inserting the entire dataset/dataframe in batches
batch_size = 10000

for start in range(0, len(df), batch_size):
    batch = df.iloc[start:start + batch_size]
    records = batch.to_dict("records")
    collection.insert_many(records)
    print("Inserted:", start + len(batch))

In [ ]:
#checking insertion succes
print(collection.count_documents({}))

In [ ]:
#PART 2
#finding the peak consumption
peak = collection.find_one(
    sort=[("Global_active_power", -1)]
)

print("Timestamp:", peak["timestamp"])
print("Power:", peak["Global_active_power"])

In [ ]:
#fetch all data points for February 2nd, 2008.
from datetime import datetime

start = datetime(2008, 2, 2)
end = datetime(2008, 2, 3)
records = list(
    collection.find(
        {
            "timestamp": {
                "$gte": start,
                "$lt": end
            }
        }
    )
)

print("Records found:", len(records))

#Since the original dataset contained missing values were removed , 
#the count is slightly lower than the USUAL 1440 minute-level measurements for a  day.

In [ ]:
#Finding records where global_active_power>7, using query by val
records = list(
    collection.find(
        {
            "Global_active_power": {
                "$gt": 7
            }
        }
    )
)
print("Count:", len(records))

In [ ]:
#Basic statistics
pipeline = [
    {
        "$group": {
            "_id": None,

            "average_power": {
                "$avg": "$Global_active_power"
            },

            "minimum_power": {
                "$min": "$Global_active_power"
            },

            "maximum_power": {
                "$max": "$Global_active_power"
            }
        }
    }
]

result = list(collection.aggregate(pipeline))
print(result)

In [ ]:
#Monthly consumption since of each month in 2008
from datetime import datetime

pipeline = [

    {
        "$match": {
            "timestamp": {
                "$gte": datetime(2008, 1, 1),
                "$lt": datetime(2009, 1, 1)
            }
        }
    },

    {
        "$group": {
            "_id": {
                "$month": "$timestamp"
            },

            "total_power": {
                "$sum": "$Global_active_power"
            }
        }
    },

    {
        "$sort": {
            "_id": 1
        }
    }

]

result = list(collection.aggregate(pipeline))

for month in result:
    print(month)

In [ ]:
import os

print(os.getcwd())